> # ⚠️ ARCHIVED — DO NOT CITE ANY NUMBER FROM THIS NOTEBOOK
>
> This notebook is from the project's first generation (2026-08-11). It is kept
> to show the methodological path, **not** as evidence.
>
> - Its stored outputs have been **stripped**, deliberately, so no figure or table
>   here can be mistaken for a current result. Git history retains them.
> - It reads data paths and episode identifiers that **no longer exist**, so it
>   cannot be re-executed to regenerate them.
> - Where it uses the Grand Ouest reference, note that the reference has since been
>   re-resolved onto a new episode reconstruction, and the linkage methods,
>   thresholds, and splits all changed afterwards. Same source data, different
>   everything else.
>
> Current evidence lives in `notebooks/10`–`14`, the `*.md` reports at the
> repository root, and `reports/boamp_methodology_chapter.pdf`.


# BOAMP Full Raw Dataset EDA

## tl;dr

This notebook runs a reproducible exploratory data analysis on the canonical raw BOAMP CSV built from the 2015-2025 yearly JSONL files. It is intentionally acquisition-safe: the raw dataset is not cleaned, deduplicated, normalized, CPV-filtered, region-filtered, or contract-matched. The notebook only computes aggregate diagnostics and visual summaries from the full raw file.

The executed notebook writes plots and summary tables into `full raw EDA/` for reproducibility.

## Context & Methods

### Key Assumptions

- Source file: `data/raw/boamp/boamp_2015_2025_raw.csv`.
- Historical interval: `2015-01-01 <= dateparution < 2026-01-01`.
- The CSV has one row per raw BOAMP notice and uses the BOAMP metadata field order.
- Empty CSV cells represent source `null`; list/object cells are compact JSON strings from the raw CSV conversion.
- All computations are streamed in chunks so the full 14GB file does not need to fit in memory.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    import plotly.io as pio
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "axes.titlesize": 15,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.titleweight": "bold",
    "axes.edgecolor": "#3b3f45",
    "axes.labelcolor": "#252a31",
    "text.color": "#252a31",
    "font.family": "DejaVu Sans",
})

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data/raw/boamp/boamp_2015_2025_raw.csv").exists():
            return candidate
    return Path.cwd()

PROJECT_ROOT = find_project_root()
CSV_PATH = PROJECT_ROOT / "data/raw/boamp/boamp_2015_2025_raw.csv"
DOWNLOAD_SUMMARY_PATH = PROJECT_ROOT / "data/metadata/boamp_download_summary.json"
CSV_SUMMARY_PATH = PROJECT_ROOT / "data/metadata/boamp_raw_csv_summary.json"
FIELDS_PATH = PROJECT_ROOT / "data/metadata/boamp_fields.json"
OUTPUT_DIR = PROJECT_ROOT / "full raw EDA"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
INTERACTIVE_DIR = OUTPUT_DIR / "interactive"

RUN_FULL_EDA = True
CHUNKSIZE = 100_000
START_YEAR = 2015
END_YEAR = 2025
HISTORICAL_START = "2015-01-01"
HISTORICAL_END_EXCLUSIVE = "2026-01-01"

# Working definition for this internship scope: Grand Ouest = Bretagne + Pays de la Loire + Normandie.
GRAND_OUEST_REGION_BY_DEPARTMENT = {
    "22": "Bretagne", "29": "Bretagne", "35": "Bretagne", "56": "Bretagne",
    "44": "Pays de la Loire", "49": "Pays de la Loire", "53": "Pays de la Loire", "72": "Pays de la Loire", "85": "Pays de la Loire",
    "14": "Normandie", "27": "Normandie", "50": "Normandie", "61": "Normandie", "76": "Normandie",
}
GRAND_OUEST_DEPARTMENTS = set(GRAND_OUEST_REGION_BY_DEPARTMENT)
GRAND_OUEST_LABEL = "Grand Ouest: Bretagne + Pays de la Loire + Normandie"

for directory in [OUTPUT_DIR, TABLE_DIR, FIGURE_DIR, INTERACTIVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

assert CSV_PATH.exists(), f"Missing canonical raw CSV: {CSV_PATH}"
assert FIELDS_PATH.exists(), f"Missing BOAMP field metadata: {FIELDS_PATH}"
print(f"CSV: {CSV_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run full EDA: {RUN_FULL_EDA}")

## Data

### 1. Load Source Metadata

These metadata files come from the acquisition pipeline and are used only to document the source and validate row counts/date coverage.

In [ ]:
fields = json.loads(FIELDS_PATH.read_text(encoding="utf-8"))
field_names = [field["name"] for field in fields]

download_summary = json.loads(DOWNLOAD_SUMMARY_PATH.read_text(encoding="utf-8")) if DOWNLOAD_SUMMARY_PATH.exists() else {}
csv_summary = json.loads(CSV_SUMMARY_PATH.read_text(encoding="utf-8")) if CSV_SUMMARY_PATH.exists() else {}

metadata_overview = pd.DataFrame({
    "metric": [
        "csv_file",
        "csv_size_gb",
        "metadata_field_count",
        "expected_rows_from_download_summary",
        "historical_start",
        "historical_end_exclusive",
    ],
    "value": [
        str(CSV_PATH),
        round(CSV_PATH.stat().st_size / 1024**3, 2),
        len(field_names),
        download_summary.get("global_validation", {}).get("historical_downloaded_count"),
        HISTORICAL_START,
        HISTORICAL_END_EXCLUSIVE,
    ],
})
metadata_overview

### Raw CSV Preview: First 20 Rows

This cell reads only the first 20 rows from the canonical raw BOAMP CSV and displays them as a dataframe. It is a preview only; the full EDA below still streams the complete dataset in chunks.

In [ ]:
raw_boamp_df = pd.read_csv(
    CSV_PATH,
    nrows=20,
    dtype="string",
    encoding="utf-8",
    low_memory=False,
)
print(f"Raw preview shape: {raw_boamp_df.shape}")
display(raw_boamp_df.head(20))

### 2. Define Streaming EDA Helpers

The full file is read with `pandas.read_csv(..., chunksize=...)`. Aggregations are accumulated into counters and compact data frames.

In [ ]:
EDA_COLUMNS = [
    "idweb",
    "dateparution",
    "datefindiffusion",
    "datelimitereponse",
    "famille",
    "famille_libelle",
    "type_procedure",
    "procedure_libelle",
    "nature",
    "nature_libelle",
    "type_marche",
    "type_avis",
    "code_departement",
    "code_departement_prestation",
    "nomacheteur",
    "titulaire",
    "source_schema",
]
CORE_FIELDS = [
    "idweb",
    "id",
    "objet",
    "dateparution",
    "nomacheteur",
    "famille_libelle",
    "procedure_libelle",
    "nature_libelle",
    "type_marche",
    "code_departement_prestation",
    "source_schema",
    "url_avis",
]
CORE_FIELDS = [field for field in CORE_FIELDS if field in field_names]
EDA_COLUMNS = list(dict.fromkeys([column for column in EDA_COLUMNS if column in field_names] + CORE_FIELDS))

LIST_LIKE_RE = re.compile(r"^\s*\[")

def values_from_cell(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if not isinstance(value, str):
        return [str(value)]
    stripped = value.strip()
    if stripped == "":
        return []
    if LIST_LIKE_RE.match(stripped):
        try:
            parsed = json.loads(stripped)
            if isinstance(parsed, list):
                return [str(item) for item in parsed if item not in [None, ""]]
        except json.JSONDecodeError:
            return [stripped]
    return [stripped]


def update_counter_from_series(counter: Counter, series: pd.Series, multi_value: bool = False):
    if multi_value:
        for value in series.dropna():
            counter.update(values_from_cell(value))
    else:
        counter.update(value for value in series.dropna().astype(str) if value.strip())


def counter_to_frame(counter: Counter, name: str, value_name: str = "records", top_n: int | None = None) -> pd.DataFrame:
    items = counter.most_common(top_n) if top_n else counter.most_common()
    frame = pd.DataFrame(items, columns=[name, value_name])
    if not frame.empty:
        frame[value_name] = frame[value_name].astype(int)
    return frame


def save_table(frame: pd.DataFrame, filename: str) -> Path:
    path = TABLE_DIR / filename
    frame.to_csv(path, index=False, encoding="utf-8")
    return path


def save_figure(fig, filename: str) -> Path:
    path = FIGURE_DIR / filename
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    return path

def normalize_department_code(value) -> str | None:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    text = str(value).strip().upper()
    if text == "" or text == "<NA>" or text == "NAN":
        return None
    return text.zfill(2) if text.isdigit() and len(text) == 1 else text


def departments_from_cell(value) -> list[str]:
    return [department for department in (normalize_department_code(item) for item in values_from_cell(value)) if department]


def first_grand_ouest_department(prestation_value, publication_value) -> str | None:
    prestation = normalize_department_code(prestation_value)
    if prestation in GRAND_OUEST_DEPARTMENTS:
        return prestation
    for department in departments_from_cell(publication_value):
        if department in GRAND_OUEST_DEPARTMENTS:
            return department
    return None


def grand_ouest_department_to_region(department: str | None) -> str | None:
    return GRAND_OUEST_REGION_BY_DEPARTMENT.get(department) if department else None


### 3. Stream the Full CSV

This cell is the main full-dataset pass. It validates date coverage, counts duplicates/missing IDs, computes yearly/monthly volumes, missingness, and top categorical dimensions.

In [ ]:
if not RUN_FULL_EDA:
    raise RuntimeError("Set RUN_FULL_EDA = True to run the full raw dataset EDA.")

started_at = datetime.now()
row_count = 0
seen_idweb = set()
duplicate_idweb = 0
missing_idweb = 0
missing_counts = Counter()
yearly_counts = Counter()
monthly_counts = Counter()
daily_counts = Counter()
family_year_counts = Counter()
family_counts = Counter()
procedure_counts = Counter()
nature_counts = Counter()
type_marche_counts = Counter()
type_avis_counts = Counter()
department_counts = Counter()
department_publication_counts = Counter()
buyer_counts = Counter()
source_schema_counts = Counter()
grand_ouest_rows = 0
grand_ouest_yearly_counts = Counter()
grand_ouest_monthly_counts = Counter()
grand_ouest_department_counts = Counter()
grand_ouest_region_counts = Counter()
grand_ouest_family_counts = Counter()
grand_ouest_procedure_counts = Counter()
grand_ouest_nature_counts = Counter()
grand_ouest_buyer_counts = Counter()
deadline_days = []
min_dateparution = None
max_dateparution = None
outside_historical_range = 0
invalid_dateparution = 0

reader = pd.read_csv(
    CSV_PATH,
    usecols=EDA_COLUMNS,
    chunksize=CHUNKSIZE,
    dtype="string",
    encoding="utf-8",
    low_memory=False,
)

for chunk_index, chunk in enumerate(reader, start=1):
    row_count += len(chunk)

    # Missingness across all EDA columns, preserving the raw CSV as-is.
    blank_like = chunk.isna() | chunk.eq("")
    missing_counts.update(blank_like.sum().astype(int).to_dict())

    idweb = chunk["idweb"] if "idweb" in chunk else pd.Series([], dtype="string")
    idweb_missing_mask = idweb.isna() | idweb.eq("")
    missing_idweb += int(idweb_missing_mask.sum())
    for value in idweb[~idweb_missing_mask].astype(str):
        if value in seen_idweb:
            duplicate_idweb += 1
        else:
            seen_idweb.add(value)

    date_series = pd.to_datetime(chunk["dateparution"], errors="coerce")
    invalid_dateparution += int(date_series.isna().sum())
    valid_dates = date_series.dropna()
    if len(valid_dates):
        chunk_min = valid_dates.min().strftime("%Y-%m-%d")
        chunk_max = valid_dates.max().strftime("%Y-%m-%d")
        min_dateparution = chunk_min if min_dateparution is None else min(min_dateparution, chunk_min)
        max_dateparution = chunk_max if max_dateparution is None else max(max_dateparution, chunk_max)

        years = valid_dates.dt.year.astype(str)
        months = valid_dates.dt.to_period("M").astype(str)
        days = valid_dates.dt.strftime("%Y-%m-%d")
        yearly_counts.update(years)
        monthly_counts.update(months)
        daily_counts.update(days)

        outside_mask = (valid_dates < pd.Timestamp(HISTORICAL_START)) | (valid_dates >= pd.Timestamp(HISTORICAL_END_EXCLUSIVE))
        outside_historical_range += int(outside_mask.sum())

        if "famille_libelle" in chunk:
            temp = pd.DataFrame({"year": years.values, "famille_libelle": chunk.loc[valid_dates.index, "famille_libelle"].fillna("Unknown").astype(str).values})
            family_year_counts.update(map(tuple, temp[["year", "famille_libelle"]].itertuples(index=False, name=None)))

    if "datelimitereponse" in chunk:
        deadline = pd.to_datetime(chunk["datelimitereponse"], errors="coerce", utc=True)
        publication = pd.to_datetime(chunk["dateparution"], errors="coerce", utc=True)
        delta = (deadline - publication).dt.total_seconds() / 86400
        usable_delta = delta.dropna()
        deadline_days.extend(usable_delta[(usable_delta >= -30) & (usable_delta <= 365)].astype(float).tolist())

    if "famille_libelle" in chunk:
        update_counter_from_series(family_counts, chunk["famille_libelle"])
    if "procedure_libelle" in chunk:
        update_counter_from_series(procedure_counts, chunk["procedure_libelle"])
    if "nature_libelle" in chunk:
        update_counter_from_series(nature_counts, chunk["nature_libelle"])
    if "type_marche" in chunk:
        update_counter_from_series(type_marche_counts, chunk["type_marche"], multi_value=True)
    if "type_avis" in chunk:
        update_counter_from_series(type_avis_counts, chunk["type_avis"], multi_value=True)
    if "code_departement_prestation" in chunk:
        update_counter_from_series(department_counts, chunk["code_departement_prestation"])
    if "code_departement" in chunk:
        update_counter_from_series(department_publication_counts, chunk["code_departement"], multi_value=True)
    if "nomacheteur" in chunk:
        update_counter_from_series(buyer_counts, chunk["nomacheteur"])
    if "source_schema" in chunk:
        update_counter_from_series(source_schema_counts, chunk["source_schema"])

    grand_ouest_department_series = pd.Series(
        [
            first_grand_ouest_department(prestation, publication)
            for prestation, publication in zip(
                chunk.get("code_departement_prestation", pd.Series(pd.NA, index=chunk.index)),
                chunk.get("code_departement", pd.Series(pd.NA, index=chunk.index)),
            )
        ],
        index=chunk.index,
        dtype="string",
    )
    grand_ouest_mask = grand_ouest_department_series.notna()
    if grand_ouest_mask.any():
        grand_ouest_rows += int(grand_ouest_mask.sum())
        grand_ouest_department_counts.update(grand_ouest_department_series[grand_ouest_mask].astype(str))
        grand_ouest_region_counts.update(
            grand_ouest_department_series[grand_ouest_mask].map(grand_ouest_department_to_region).dropna().astype(str)
        )

        grand_ouest_valid_dates = date_series.loc[grand_ouest_mask].dropna()
        if len(grand_ouest_valid_dates):
            grand_ouest_yearly_counts.update(grand_ouest_valid_dates.dt.year.astype(str))
            grand_ouest_monthly_counts.update(grand_ouest_valid_dates.dt.to_period("M").astype(str))

        grand_ouest_chunk = chunk.loc[grand_ouest_mask]
        if "famille_libelle" in grand_ouest_chunk:
            update_counter_from_series(grand_ouest_family_counts, grand_ouest_chunk["famille_libelle"])
        if "procedure_libelle" in grand_ouest_chunk:
            update_counter_from_series(grand_ouest_procedure_counts, grand_ouest_chunk["procedure_libelle"])
        if "nature_libelle" in grand_ouest_chunk:
            update_counter_from_series(grand_ouest_nature_counts, grand_ouest_chunk["nature_libelle"])
        if "nomacheteur" in grand_ouest_chunk:
            update_counter_from_series(grand_ouest_buyer_counts, grand_ouest_chunk["nomacheteur"])

    if chunk_index % 5 == 0:
        print(f"Processed {row_count:,} rows across {chunk_index} chunks...")

finished_at = datetime.now()
print(f"Finished streaming {row_count:,} rows in {(finished_at - started_at).total_seconds() / 60:.1f} minutes")

## Results

### 4. Validation Snapshot

These checks verify that the EDA used the full historical raw dataset and did not accidentally include 2026 records.

In [ ]:
validation_snapshot = pd.DataFrame({
    "metric": [
        "rows_streamed",
        "expected_rows_from_download_summary",
        "counts_match_download_summary",
        "unique_idweb",
        "duplicate_idweb",
        "missing_idweb",
        "min_dateparution",
        "max_dateparution",
        "outside_historical_range",
        "invalid_dateparution",
    ],
    "value": [
        row_count,
        download_summary.get("global_validation", {}).get("historical_downloaded_count"),
        row_count == download_summary.get("global_validation", {}).get("historical_downloaded_count"),
        len(seen_idweb),
        duplicate_idweb,
        missing_idweb,
        min_dateparution,
        max_dateparution,
        outside_historical_range,
        invalid_dateparution,
    ],
})
save_table(validation_snapshot, "validation_snapshot.csv")
validation_snapshot

In [ ]:
yearly_df = pd.DataFrame(sorted(yearly_counts.items()), columns=["year", "records"])
yearly_df["year"] = yearly_df["year"].astype(int)
yearly_df["share"] = yearly_df["records"] / yearly_df["records"].sum()
monthly_df = pd.DataFrame(sorted(monthly_counts.items()), columns=["month", "records"])
monthly_df["month"] = pd.to_datetime(monthly_df["month"])
monthly_df["year"] = monthly_df["month"].dt.year
monthly_df["month_number"] = monthly_df["month"].dt.month
monthly_df["month_name"] = monthly_df["month"].dt.strftime("%b")
daily_df = pd.DataFrame(sorted(daily_counts.items()), columns=["date", "records"])
daily_df["date"] = pd.to_datetime(daily_df["date"])

family_df = counter_to_frame(family_counts, "famille_libelle")
procedure_df = counter_to_frame(procedure_counts, "procedure_libelle")
nature_df = counter_to_frame(nature_counts, "nature_libelle")
type_marche_df = counter_to_frame(type_marche_counts, "type_marche")
type_avis_df = counter_to_frame(type_avis_counts, "type_avis")
department_df = counter_to_frame(department_counts, "code_departement_prestation")
department_publication_df = counter_to_frame(department_publication_counts, "code_departement")
buyer_df = counter_to_frame(buyer_counts, "nomacheteur")
source_schema_df = counter_to_frame(source_schema_counts, "source_schema")

for frame, name in [
    (yearly_df, "yearly_counts.csv"),
    (monthly_df, "monthly_counts.csv"),
    (daily_df, "daily_counts.csv"),
    (family_df, "family_counts.csv"),
    (procedure_df, "procedure_counts.csv"),
    (nature_df, "nature_counts.csv"),
    (type_marche_df, "type_marche_counts.csv"),
    (type_avis_df, "type_avis_counts.csv"),
    (department_df, "department_prestation_counts.csv"),
    (department_publication_df, "department_publication_counts.csv"),
    (buyer_df, "buyer_counts.csv"),
    (source_schema_df, "source_schema_counts.csv"),
]:
    save_table(frame, name)

yearly_df

### 5. Publication Volume by Year

A line-and-bar chart shows the long-run volume of raw BOAMP notices in the historical corpus.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8))
bar_color = "#4E79A7"
line_color = "#F28E2B"
ax.bar(yearly_df["year"], yearly_df["records"], color=bar_color, alpha=0.82, edgecolor="#2f465f", linewidth=0.7)
ax.plot(yearly_df["year"], yearly_df["records"], color=line_color, marker="o", linewidth=2.5)
for _, row in yearly_df.iterrows():
    ax.text(row["year"], row["records"] + yearly_df["records"].max() * 0.012, f"{row['records']/1000:.0f}k", ha="center", va="bottom", fontsize=9)
ax.set_title("BOAMP raw notices by publication year")
ax.set_subtitle = None
ax.set_xlabel("Publication year")
ax.set_ylabel("Records")
ax.set_xticks(yearly_df["year"])
ax.yaxis.set_major_formatter(lambda x, pos: f"{x/1000:.0f}k")
sns.despine(ax=ax)
save_figure(fig, "01_publication_volume_by_year.png")
plt.show()

### 6. Monthly Seasonality Heatmap

This heatmap makes seasonal publication patterns easier to scan across all complete calendar years.

In [ ]:
month_order = list(range(1, 13))
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
heatmap_data = monthly_df.pivot_table(index="year", columns="month_number", values="records", aggfunc="sum").reindex(columns=month_order)
fig, ax = plt.subplots(figsize=(12, 6.5))
sns.heatmap(
    heatmap_data,
    cmap=sns.light_palette("#4E79A7", as_cmap=True),
    linewidths=0.5,
    linecolor="white",
    annot=True,
    fmt=".0f",
    annot_kws={"fontsize": 8},
    cbar_kws={"label": "Records"},
    ax=ax,
)
ax.set_title("BOAMP monthly publication volume heatmap")
ax.set_xlabel("Publication month")
ax.set_ylabel("Publication year")
ax.set_xticklabels(month_labels, rotation=0)
save_figure(fig, "02_monthly_publication_heatmap.png")
plt.show()

### 7. Notice Family Mix Over Time

The chart keeps the largest BOAMP families visible and groups smaller families as `Other` only for visualization readability. The source records remain unchanged.

In [ ]:
family_year_df = pd.DataFrame(
    [(year, family, count) for (year, family), count in family_year_counts.items()],
    columns=["year", "famille_libelle", "records"],
)
family_year_df["year"] = family_year_df["year"].astype(int)
top_families = family_df.head(6)["famille_libelle"].tolist()
family_year_df["family_for_plot"] = np.where(family_year_df["famille_libelle"].isin(top_families), family_year_df["famille_libelle"], "Other")
family_plot_df = family_year_df.groupby(["year", "family_for_plot"], as_index=False)["records"].sum()
family_share_df = family_plot_df.merge(yearly_df[["year", "records"]].rename(columns={"records": "year_total"}), on="year")
family_share_df["share"] = family_share_df["records"] / family_share_df["year_total"]
family_share_pivot = family_share_df.pivot_table(index="year", columns="family_for_plot", values="share", aggfunc="sum").fillna(0)
ordered_columns = [family for family in top_families if family in family_share_pivot.columns] + (["Other"] if "Other" in family_share_pivot.columns else [])
family_share_pivot = family_share_pivot[ordered_columns]
save_table(family_share_df, "family_share_by_year.csv")

palette = ["#4E79A7", "#F28E2B", "#59A14F", "#E15759", "#B07AA1", "#EDC948", "#BAB0AC"]
fig, ax = plt.subplots(figsize=(12, 6.5))
ax.stackplot(family_share_pivot.index, [family_share_pivot[col] for col in family_share_pivot.columns], labels=family_share_pivot.columns, colors=palette[:len(family_share_pivot.columns)], alpha=0.9)
ax.set_title("BOAMP notice family composition by publication year")
ax.set_xlabel("Publication year")
ax.set_ylabel("Share of records")
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
ax.set_xlim(START_YEAR, END_YEAR)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
sns.despine(ax=ax)
save_figure(fig, "03_notice_family_share_by_year.png")
plt.show()

### 8. Top Procedures, Natures, and Market Types

These horizontal bars summarize the most common procurement descriptors in the raw BOAMP notices.

In [ ]:
def plot_top_bar(frame, label_col, value_col, title, filename, color="#4E79A7", top_n=15):
    data = frame.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, max(5, top_n * 0.35)))
    ax.barh(data[label_col], data[value_col], color=color, alpha=0.86, edgecolor="#2f465f", linewidth=0.6)
    ax.set_title(title)
    ax.set_xlabel("Records")
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(lambda x, pos: f"{x/1000:.0f}k")
    for y, value in enumerate(data[value_col]):
        ax.text(value, y, f" {value:,}", va="center", fontsize=9)
    sns.despine(ax=ax, left=True)
    save_figure(fig, filename)
    plt.show()

plot_top_bar(procedure_df, "procedure_libelle", "records", "Top BOAMP procedures", "04_top_procedures.png", color="#4E79A7")
plot_top_bar(nature_df, "nature_libelle", "records", "Top BOAMP notice natures", "05_top_notice_natures.png", color="#59A14F")
plot_top_bar(type_marche_df, "type_marche", "records", "Top BOAMP market types", "06_top_market_types.png", color="#F28E2B", top_n=10)

### 9. Geographic Coverage Proxy

BOAMP contains department codes rather than geometry in the raw CSV. This ranking uses `code_departement_prestation` as a simple geographic coverage proxy.

In [ ]:
plot_top_bar(department_df, "code_departement_prestation", "records", "Top departments by prestation code", "07_top_departments_prestation.png", color="#B07AA1", top_n=25)

### 10. Grand Ouest Regional EDA

For the internship scope, this notebook defines **Grand Ouest** as **Bretagne + Pays de la Loire + Normandie**. A record is included when either `code_departement_prestation` or, if needed, the publication department list `code_departement` contains one of these departments.

This is a raw regional EDA slice. It does not clean, deduplicate, classify digital contracts, or infer renewals.

In [ ]:
grand_ouest_yearly_df = pd.DataFrame(sorted(grand_ouest_yearly_counts.items()), columns=["year", "grand_ouest_records"])
grand_ouest_yearly_df["year"] = grand_ouest_yearly_df["year"].astype(int)
grand_ouest_yearly_df = yearly_df[["year", "records"]].merge(grand_ouest_yearly_df, on="year", how="left").fillna({"grand_ouest_records": 0})
grand_ouest_yearly_df["grand_ouest_records"] = grand_ouest_yearly_df["grand_ouest_records"].astype(int)
grand_ouest_yearly_df["grand_ouest_share_of_france"] = grand_ouest_yearly_df["grand_ouest_records"] / grand_ouest_yearly_df["records"]

grand_ouest_monthly_df = pd.DataFrame(sorted(grand_ouest_monthly_counts.items()), columns=["month", "records"])
grand_ouest_monthly_df["month"] = pd.to_datetime(grand_ouest_monthly_df["month"])
grand_ouest_monthly_df["year"] = grand_ouest_monthly_df["month"].dt.year
grand_ouest_monthly_df["month_number"] = grand_ouest_monthly_df["month"].dt.month

grand_ouest_department_df = counter_to_frame(grand_ouest_department_counts, "department", "records")
grand_ouest_department_df["region"] = grand_ouest_department_df["department"].map(GRAND_OUEST_REGION_BY_DEPARTMENT)
grand_ouest_region_df = counter_to_frame(grand_ouest_region_counts, "region", "records")
grand_ouest_family_df = counter_to_frame(grand_ouest_family_counts, "famille_libelle", "records")
grand_ouest_procedure_df = counter_to_frame(grand_ouest_procedure_counts, "procedure_libelle", "records")
grand_ouest_nature_df = counter_to_frame(grand_ouest_nature_counts, "nature_libelle", "records")
grand_ouest_buyer_df = counter_to_frame(grand_ouest_buyer_counts, "nomacheteur", "records")

grand_ouest_overview = pd.DataFrame([
    ["Definition", GRAND_OUEST_LABEL],
    ["Departments", ", ".join(sorted(GRAND_OUEST_DEPARTMENTS))],
    ["Grand Ouest records", f"{grand_ouest_rows:,}"],
    ["France records", f"{row_count:,}"],
    ["Grand Ouest share of France", f"{grand_ouest_rows / row_count:.1%}"],
], columns=["metric", "value"])

for frame, name in [
    (grand_ouest_overview, "grand_ouest_overview.csv"),
    (grand_ouest_yearly_df, "grand_ouest_yearly_counts.csv"),
    (grand_ouest_monthly_df, "grand_ouest_monthly_counts.csv"),
    (grand_ouest_department_df, "grand_ouest_department_counts.csv"),
    (grand_ouest_region_df, "grand_ouest_region_counts.csv"),
    (grand_ouest_family_df, "grand_ouest_family_counts.csv"),
    (grand_ouest_procedure_df, "grand_ouest_procedure_counts.csv"),
    (grand_ouest_nature_df, "grand_ouest_nature_counts.csv"),
    (grand_ouest_buyer_df, "grand_ouest_buyer_counts.csv"),
]:
    save_table(frame, name)

display(grand_ouest_overview)
display(grand_ouest_yearly_df)
display(grand_ouest_region_df)

### 11. Grand Ouest Volume and Mix Visuals

These figures compare Grand Ouest volume over time, department/region mix, seasonal publication patterns, and the leading raw procurement descriptors in the regional subset.

In [ ]:
fig, ax1 = plt.subplots(figsize=(11.5, 5.8))
ax1.bar(grand_ouest_yearly_df["year"], grand_ouest_yearly_df["grand_ouest_records"], color="#4E79A7", alpha=0.84, edgecolor="#2f465f", linewidth=0.7)
ax1.set_title("BOAMP raw notices in Grand Ouest by publication year")
ax1.set_xlabel("Publication year")
ax1.set_ylabel("Grand Ouest records")
ax1.set_xticks(grand_ouest_yearly_df["year"])
ax1.yaxis.set_major_formatter(lambda x, pos: f"{x/1000:.0f}k")
for _, row in grand_ouest_yearly_df.iterrows():
    ax1.text(row["year"], row["grand_ouest_records"] + grand_ouest_yearly_df["grand_ouest_records"].max() * 0.012, f"{row['grand_ouest_records']/1000:.0f}k", ha="center", va="bottom", fontsize=9)
ax2 = ax1.twinx()
ax2.plot(grand_ouest_yearly_df["year"], grand_ouest_yearly_df["grand_ouest_share_of_france"], color="#F28E2B", marker="o", linewidth=2.3)
ax2.set_ylabel("Share of France")
ax2.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
sns.despine(ax=ax1, right=False)
save_figure(fig, "13_grand_ouest_volume_by_year.png")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8), gridspec_kw={"width_ratios": [1, 1.45]})
region_plot = grand_ouest_region_df.iloc[::-1]
axes[0].barh(region_plot["region"], region_plot["records"], color="#59A14F", alpha=0.86, edgecolor="#365f34", linewidth=0.6)
axes[0].set_title("Grand Ouest records by region")
axes[0].set_xlabel("Records")
axes[0].xaxis.set_major_formatter(lambda x, pos: f"{x/1000:.0f}k")
for y, value in enumerate(region_plot["records"]):
    axes[0].text(value, y, f" {value:,}", va="center", fontsize=9)

department_plot = grand_ouest_department_df.sort_values("records", ascending=True)
axes[1].barh(department_plot["department"], department_plot["records"], color="#B07AA1", alpha=0.86, edgecolor="#6e4966", linewidth=0.6)
axes[1].set_title("Grand Ouest records by department")
axes[1].set_xlabel("Records")
axes[1].set_ylabel("Department")
axes[1].xaxis.set_major_formatter(lambda x, pos: f"{x/1000:.0f}k")
for y, value in enumerate(department_plot["records"]):
    axes[1].text(value, y, f" {value:,}", va="center", fontsize=8)
sns.despine(fig=fig, left=True)
save_figure(fig, "14_grand_ouest_region_department_mix.png")
plt.show()

if not grand_ouest_monthly_df.empty:
    go_heatmap_data = grand_ouest_monthly_df.pivot_table(index="year", columns="month_number", values="records", aggfunc="sum").reindex(columns=month_order)
    fig, ax = plt.subplots(figsize=(12, 6.5))
    sns.heatmap(
        go_heatmap_data,
        cmap=sns.light_palette("#59A14F", as_cmap=True),
        linewidths=0.5,
        linecolor="white",
        annot=True,
        fmt=".0f",
        annot_kws={"fontsize": 8},
        cbar_kws={"label": "Records"},
        ax=ax,
    )
    ax.set_title("Grand Ouest monthly BOAMP publication volume heatmap")
    ax.set_xlabel("Publication month")
    ax.set_ylabel("Publication year")
    ax.set_xticklabels(month_labels, rotation=0)
    save_figure(fig, "15_grand_ouest_monthly_heatmap.png")
    plt.show()

plot_top_bar(grand_ouest_family_df, "famille_libelle", "records", "Top BOAMP families in Grand Ouest", "16_grand_ouest_top_families.png", color="#4E79A7", top_n=12)
plot_top_bar(grand_ouest_procedure_df, "procedure_libelle", "records", "Top BOAMP procedures in Grand Ouest", "17_grand_ouest_top_procedures.png", color="#F28E2B", top_n=12)
plot_top_bar(grand_ouest_buyer_df, "nomacheteur", "records", "Top buyers in Grand Ouest", "18_grand_ouest_top_buyers.png", color="#B07AA1", top_n=20)

### 10. Buyer Concentration

A log-scale ranking helps make concentration visible without hiding the long tail of buyers.

In [ ]:
top_buyers = buyer_df.head(25).iloc[::-1]
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(top_buyers["nomacheteur"], top_buyers["records"], color="#4E79A7", alpha=0.86, edgecolor="#2f465f", linewidth=0.6)
ax.set_xscale("log")
ax.set_title("Top buyers by raw BOAMP notice count")
ax.set_xlabel("Records, log scale")
ax.set_ylabel("")
for y, value in enumerate(top_buyers["records"]):
    ax.text(value, y, f" {value:,}", va="center", fontsize=8)
sns.despine(ax=ax, left=True)
save_figure(fig, "08_top_buyers_log_scale.png")
plt.show()

### 11. Core Field Missingness

This is a raw-data completeness diagnostic. Missing values are reported only; no rows or fields are removed.

In [ ]:
missing_df = pd.DataFrame(
    [{"field": field, "missing_records": int(missing_counts.get(field, 0)), "missing_share": int(missing_counts.get(field, 0)) / row_count} for field in EDA_COLUMNS]
).sort_values("missing_share", ascending=False)
core_missing_df = missing_df[missing_df["field"].isin(CORE_FIELDS)].sort_values("missing_share", ascending=True)
save_table(missing_df, "missingness_analyzed_fields.csv")

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.barh(core_missing_df["field"], core_missing_df["missing_share"], color="#E15759", alpha=0.82, edgecolor="#8f3032", linewidth=0.6)
ax.set_title("Missingness in core BOAMP raw fields")
ax.set_xlabel("Share of records with missing or empty value")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")
for y, value in enumerate(core_missing_df["missing_share"]):
    ax.text(value, y, f" {value:.1%}", va="center", fontsize=9)
sns.despine(ax=ax, left=True)
save_figure(fig, "09_core_field_missingness.png")
plt.show()
missing_df.head(12)

### 12. Days From Publication to Response Deadline

This distribution is derived from `datelimitereponse - dateparution` for records where both raw fields parse as dates. The plotted range is bounded for readability and the raw source file is unchanged.

In [ ]:
deadline_days_series = pd.Series(deadline_days, name="days_to_deadline")
deadline_summary = deadline_days_series.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame().reset_index().rename(columns={"index": "statistic"})
save_table(deadline_summary, "deadline_days_summary.csv")

fig, ax = plt.subplots(figsize=(11, 5.8))
plot_values = deadline_days_series[(deadline_days_series >= 0) & (deadline_days_series <= 120)]
sns.histplot(plot_values, bins=60, color="#4E79A7", edgecolor="white", linewidth=0.3, ax=ax)
ax.axvline(plot_values.median(), color="#F28E2B", linewidth=2.2, label=f"Median: {plot_values.median():.0f} days")
ax.set_title("Distribution of days from publication to response deadline")
ax.set_xlabel("Days from dateparution to datelimitereponse, bounded to 0-120 for display")
ax.set_ylabel("Records")
ax.legend(frameon=False)
ax.yaxis.set_major_formatter(lambda x, pos: f"{x/1000:.0f}k")
sns.despine(ax=ax)
save_figure(fig, "10_deadline_days_distribution.png")
plt.show()
deadline_summary

### 13. Daily Volume Distribution by Year

This box plot highlights daily publication volume variability by year.

In [ ]:
daily_df["year"] = daily_df["date"].dt.year
fig, ax = plt.subplots(figsize=(12, 6.5))
sns.boxplot(data=daily_df, x="year", y="records", color="#A0CBE8", fliersize=2, linewidth=0.9, ax=ax)
sns.stripplot(data=daily_df.sample(min(len(daily_df), 1200), random_state=42), x="year", y="records", color="#4E79A7", alpha=0.18, size=2, ax=ax)
ax.set_title("Daily BOAMP publication volume distribution by year")
ax.set_xlabel("Publication year")
ax.set_ylabel("Records per publication day")
sns.despine(ax=ax)
save_figure(fig, "11_daily_volume_distribution_by_year.png")
plt.show()

### 14. Source Schema Mix

This chart checks the distribution of source schemas present in the raw corpus.

In [ ]:
plot_top_bar(source_schema_df, "source_schema", "records", "Top BOAMP source schemas", "12_top_source_schemas.png", color="#EDC948", top_n=15)

### 15. Optional Interactive HTML Chart

If Plotly is available, save an interactive monthly trend chart as HTML alongside the static PNG files.

In [ ]:
interactive_outputs = []
if PLOTLY_AVAILABLE:
    fig = px.line(
        monthly_df,
        x="month",
        y="records",
        title="BOAMP monthly publication volume",
        markers=True,
        labels={"month": "Publication month", "records": "Records"},
        template="plotly_white",
    )
    fig.update_traces(line_color="#4E79A7", marker_color="#F28E2B")
    html_path = INTERACTIVE_DIR / "monthly_publication_volume.html"
    fig.write_html(html_path, include_plotlyjs=True)
    interactive_outputs.append(str(html_path))
interactive_outputs

## Takeaways

The notebook updates this final table from executed outputs. Use it as a compact handoff checklist for the raw EDA run.

In [ ]:
outputs = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_csv": str(CSV_PATH),
    "output_dir": str(OUTPUT_DIR),
    "rows_streamed": int(row_count),
    "expected_rows": download_summary.get("global_validation", {}).get("historical_downloaded_count"),
    "counts_match_expected": bool(row_count == download_summary.get("global_validation", {}).get("historical_downloaded_count")),
    "field_count": len(field_names),
    "unique_idweb": int(len(seen_idweb)),
    "duplicate_idweb": int(duplicate_idweb),
    "missing_idweb": int(missing_idweb),
    "min_dateparution": min_dateparution,
    "max_dateparution": max_dateparution,
    "outside_historical_range": int(outside_historical_range),
    "invalid_dateparution": int(invalid_dateparution),
    "static_figures": sorted(str(path) for path in FIGURE_DIR.glob("*.png")),
    "tables": sorted(str(path) for path in TABLE_DIR.glob("*.csv")),
    "interactive_outputs": interactive_outputs,
    "grand_ouest": {
        "definition": GRAND_OUEST_LABEL,
        "departments": sorted(GRAND_OUEST_DEPARTMENTS),
        "rows": int(grand_ouest_rows),
        "share_of_france": float(grand_ouest_rows / row_count) if row_count else None,
    },
}
(OUTPUT_DIR / "eda_run_summary.json").write_text(json.dumps(outputs, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

pd.DataFrame([
    ["Rows streamed", f"{outputs['rows_streamed']:,}"],
    ["Counts match expected", outputs["counts_match_expected"]],
    ["Date coverage", f"{outputs['min_dateparution']} to {outputs['max_dateparution']}"],
    ["Rows outside historical range", outputs["outside_historical_range"]],
    ["Duplicate idweb", f"{outputs['duplicate_idweb']:,}"],
    ["Missing idweb", f"{outputs['missing_idweb']:,}"],
    ["Grand Ouest rows", f"{outputs['grand_ouest']['rows']:,}"],
    ["Grand Ouest share", f"{outputs['grand_ouest']['share_of_france']:.1%}"],
    ["Static figures saved", len(outputs["static_figures"])],
    ["Tables saved", len(outputs["tables"])],
], columns=["check", "value"])